# 🎙️ Tamil Speech-to-Text (ASR) Deep Learning Pipeline
### தமிழ் பேச்சு-எழுத்து மாற்றி மாதிரி (Google Colab GPU Optimized)

This notebook trains an end-to-end **2D Residual-CNN + Multi-layer Bidirectional LSTM + CTC Loss** deep acoustic model to transcribe spoken Tamil audio into Unicode Tamil text characters.

#### 🌟 Key Features:
- 🧠 **Acoustic Model**: 2D-CNN feature extractor + Deep BiLSTM + CTC Projection Layer.
- 🔤 **Tamil Character Vocabulary**: Full Unicode Tamil graphemes (உயிர், மெய், உயிர்மெய் எழுத்துக்கள், ஆய்தம், இடைவெளி).
- 📊 **Metrics**: Evaluates Character Error Rate (CER) and Word Error Rate (WER).
- 🌐 **Public Web Studio**: One-click live launch with Localtunnel.

## 1. 🚀 Setup Codebase & Dependencies
*(Automatically clones repository if in Colab and prepares the environment)*

In [ ]:
import os
import sys

# Auto-clone repository if running directly in fresh Colab session
if not os.path.exists("src"):
    print("⬇️ Cloning ML-MODEL repository from GitHub...")
    !git clone https://github.com/kevinjosh10/ML-MODEL.git
    %cd ML-MODEL

# Ensure project root is in Python sys.path
project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Working directory: {os.getcwd()}")

# Install dependencies
print("📦 Installing audio and ML dependencies...")
!pip install -q torchaudio librosa soundfile matplotlib scikit-learn tqdm fastapi uvicorn python-multipart jinja2 gtts

import torch
import torchaudio
print(f"\n🔥 PyTorch Version: {torch.__version__}")
print(f"🎵 Torchaudio Version: {torchaudio.__version__}")
print(f"⚡ CUDA GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 Active GPU: {torch.cuda.get_device_name(0)}")

## 2. ⚙️ Prepare Paired Tamil Speech-to-Text Dataset
*(Generates and tokenizes authentic spoken Tamil sentences paired with Unicode text transcripts)*

In [ ]:
from src.config import Config
from src.data.vocabulary import tamil_vocab
from src.data.dataset import get_asr_data_loaders
from src.data.audio_preprocessing import AudioPreprocessor
from src.models import build_asr_model
from src.training.trainer import ASRTrainer
from src.training.metrics import evaluate_asr_model
import IPython.display as ipd
import random

# Initialize ASR configuration
config = Config(
    sample_rate=16000,
    n_mels=80,
    epochs=30,
    batch_size=8,
    learning_rate=5e-4,
    seed=42
)

print(f"🔤 Tamil Character Vocabulary Size: {config.vocab_size} tokens (Blank ID = {config.blank_id})")
train_loader, val_loader, test_loader, records = get_asr_data_loaders(config)
print(f"📊 Data Split: {len(train_loader.dataset)} Train | {len(val_loader.dataset)} Val | {len(test_loader.dataset)} Test")

## 3. 🎧 Audio Player & Spectrogram Visualizer
Listen to sample Tamil speech and view its acoustic Log-Mel frequency spectrum.

In [ ]:
import matplotlib.pyplot as plt
import librosa.display

sample_item = random.choice(records)
print(f"🔊 Tamil Audio Sample: {sample_item['audio_path']}")
print(f"📝 Ground Truth Transcript: \"{sample_item['tamil_text']}\"")
print(f"🌐 English Translation: \"{sample_item['english']}\"")

# Play Audio
ipd.display(ipd.Audio(sample_item['audio_path']))

# Plot Spectrogram
preprocessor = AudioPreprocessor(config, is_train=False)
y = preprocessor.load_audio(sample_item['audio_path']).squeeze().cpu().numpy()
mel = preprocessor.extract_mel_spectrogram(torch.from_numpy(y), augment=False)
if mel.dim() == 3: mel = mel[0]

plt.figure(figsize=(10, 3), facecolor='#0B0F19')
ax = plt.subplot(1, 1, 1)
ax.set_facecolor('#0B0F19')
librosa.display.specshow(mel.numpy(), sr=config.sample_rate, hop_length=config.hop_length, x_axis='time', y_axis='mel', cmap='magma', ax=ax)
plt.title(f"Tamil Speech Log-Mel Spectrogram: {sample_item['tamil_text'][:30]}...", color='white')
plt.tick_params(colors='white')
plt.show()

## 4. 🧠 Train the 2D-CNN + BiLSTM + CTC ASR Model
Trains the acoustic model using PyTorch `nn.CTCLoss` with GPU acceleration.

In [ ]:
model = build_asr_model(config)
print("Model Architecture:")
print(model)

trainer = ASRTrainer(model, config, train_loader, val_loader)
trainer.fit()

## 5. 📊 Evaluate Character Error Rate (CER) & Word Error Rate (WER)

In [ ]:
best_checkpoint = config.checkpoint_dir / "best_tamil_asr_model.pth"
if best_checkpoint.exists():
    ckpt = torch.load(best_checkpoint, map_location=config.device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"✅ Loaded best checkpoint (Epoch {ckpt.get('epoch', 0)+1})")

results = evaluate_asr_model(model, test_loader, config)
print(f"\n🎯 Test Character Error Rate (CER): {results['cer']*100:.2f}%")
print(f"🎯 Test Word Error Rate (WER): {results['wer']*100:.2f}%")
print(f"✨ Character Accuracy: {results['character_accuracy']:.2f}%")

print("\n📋 Sample Test Transcriptions:")
for hyp, ref in zip(results['sample_hypotheses'][:5], results['sample_references'][:5]):
    print(f"  • Reference  : \"{ref}\"")
    print(f"    Hypothesis : \"{hyp}\"")
    print("-" * 50)

from src.utils.audio_recorder import record_audio_in_colab
from src.inference import TamilASRPredictor
import random

predictor = TamilASRPredictor(model, config=config)

print("🎙️ Speak into your microphone now (allow browser mic prompt if asked):")
recorded_path = record_audio_in_colab(filename="colab_test_voice.wav", duration=4.0)

# Automatic fallback to dataset sample if mic was skipped
if not recorded_path or not os.path.exists(recorded_path):
    print("\nℹ️ Microphone was skipped or unavailable. Testing on a sample from the dataset instead...")
    sample_item = random.choice(records)
    recorded_path = sample_item["audio_path"]
    print(f"🔊 Selected Sample: {recorded_path}")
    print(f"🎯 Expected Transcript: \"{sample_item['tamil_text']}\"")

if recorded_path and os.path.exists(recorded_path):
    ipd.display(ipd.Audio(recorded_path))
    
    result = predictor.transcribe_file(recorded_path)
    print("\n" + "="*60)
    print(f"📝 Transcribed Tamil Text : {result['tamil_text']}")
    print(f"🌐 English Translation     : {result['english_translation']}")
    print(f"✨ Confidence Score        : {result['confidence_percentage']}")
    print(f"⚡ Speech Pace             : {result['words_per_minute']} WPM ({result['words_count']} words)")
    print("="*60)


In [ ]:
from src.utils.audio_recorder import record_audio_in_colab
from src.inference import TamilASRPredictor

predictor = TamilASRPredictor(model, config=config)

print("🎙️ Speak into your microphone now (or press stop):")
recorded_path = record_audio_in_colab(duration=4.0, output_path="colab_test_voice.wav")

if recorded_path and os.path.exists(recorded_path):
    print(f"\n🔊 Testing Audio Clip: {recorded_path}")
    ipd.display(ipd.Audio(recorded_path))
    
    result = predictor.transcribe_file(recorded_path)
    print("\n" + "="*60)
    print(f"📝 Transcribed Tamil Text : {result['tamil_text']}")
    print(f"🌐 English Translation     : {result['english_translation']}")
    print(f"✨ Confidence Score        : {result['confidence_percentage']}")
    print(f"⚡ Speech Pace             : {result['words_per_minute']} WPM ({result['words_count']} words)")
    print("="*60)

## 7. 🌐 Launch Live Public Web Studio from Colab
Run this cell to start the **Tamil Speech-to-Text Web Studio** and access it from any browser via Localtunnel.

In [ ]:
import subprocess
import time
import urllib.request

# 1. Start FastAPI backend with PyTorch model in background
print("🚀 Starting Tamil Speech-to-Text AI Web Server on port 8000...")
process = subprocess.Popen(["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# 2. Expose via free Localtunnel
print("\n🌐 Creating public tunnel via Localtunnel...")
!npm install -g localtunnel -q

try:
    public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print(f"🔑 Localtunnel Password (if asked for endpoint IP): {public_ip}")
except Exception:
    pass

print("\n✨ Click the link generated below to open your Live Tamil Speech-to-Text Studio:")
!npx localtunnel --port 8000
